$$
\begin{array}{c}
\textbf{DS-UA 201: CAUSAL INFERENCE}\\\\
\textbf{Lab — Control Variables}\\\\
\textit{Center for Data Science, New York University}
\end{array}
$$
---

## 1. From natural experiments to conditional independence

Last time we talked about **natural experiments**: situations where the treatment $S$ happens to be (plausibly) independent of the other causal determinants of the outcome, $U$, in the real world.

Sometimes we cannot claim $S \perp U$ outright, but we *can* claim that the treatment is independent of $U$ **conditional on some other variable(s)** $C$:

$$
S \perp U \mid C
$$

When that holds, we can use **multiple linear regression with control variable(s)** to recover estimates of causal effects.

In [1]:
import numpy as np                       
import pandas as pd                    
import matplotlib.pyplot as plt          
import statsmodels.formula.api as smf    

## 2. Independence vs. conditional independence

### Independence

Two random variables $X$ and $Y$ are independent if, for any sets $A$ and $B$,

$$
P(X \in A, Y \in B) = P(X \in A)\,P(Y \in B).
$$

Independence means there is no relationship between the two variables: knowing one tells you nothing about the other.

### Conditional independence

Sometimes we can't say $X$ and $Y$ are independent, but we can say they are independent conditional on some other variable $Z$, written $X \perp Y \mid Z$:

$$
P(X \in A, Y \in B \mid Z) = P(X \in A \mid Z)\,P(Y \in B \mid Z).
$$

That is, $X$ and $Y$ have no relationship once we account for the value taken by $Z$. Two equivalent ways of saying this:

$$
P(X \in A \mid Z, Y) = P(X \in A \mid Z)
$$

— if we already know $Z$, also knowing $Y$ doesn't change how likely $X$ is to take any particular value. And a critical implication:

$$
E[X \mid Y, Z] = E[X \mid Z]
$$

— knowing $Y$, on top of $Z$, doesn't change our "best guess" of $X$.

### A first look in code: covariance with and without conditioning

Recall that independent variables have zero covariance. The cell below builds two situations:

- two **independent** dice rolls, whose covariance should be about 0;
- two variables $Y$ and $Z$ that are both built from a common variable $X$. They are **dependent** overall (large covariance), but once we look *within* a single value of $X$, the only thing left in each is its own independent noise — so they should be **independent conditional on $X$** (covariance about 0 within each value of $X$).

In [ ]:
np.random.seed(201)

#  indep variables: two separate dice, 1,000 rolls each 
X_ind = np.random.randint(1, 7, 1000)   # die 1: integers from 1 to 6 (the upper bound 7 is excluded)
Y_ind = np.random.randint(1, 7, 1000)   # die 2: drawn separately, so it has nothing to do with die 1

cov_ind = np.cov(X_ind, Y_ind)[0, 1]    # sample covariance between the two dice
print(f"Covariance of the two independent dice: {cov_ind:.3f}")

#  dep variables that share a common cause X 
X_dep = np.random.randint(1, 10, 1000)                  # the common cause: integers from 1 to 9
Y_dep = 2 * X_dep + np.random.normal(0, 1, 1000)        # Y = 2X + its own noise
Z_dep = 2 * X_dep + np.random.normal(0, 1, 1000)        # Z = 2X + separate noise

cov_dep = np.cov(Y_dep, Z_dep)[0, 1]
print(f"Covariance of Y and Z overall: {cov_dep:.3f}")

#  same Y and Z, but now computed separately within each value of X 
for x_value in range(1, 10):
    in_group = (X_dep == x_value)                       # True for the rows where X equals x_value
    cov_within = np.cov(Y_dep[in_group], Z_dep[in_group])[0, 1]
    print(f"Covariance of Y and Z given X = {x_value}: {cov_within:.3f}")

Covariance of the two independent dice: -0.097
Covariance of Y and Z overall: 25.551
Covariance of Y and Z given X = 1: -0.170
Covariance of Y and Z given X = 2: -0.201
Covariance of Y and Z given X = 3: -0.039
Covariance of Y and Z given X = 4: 0.123
Covariance of Y and Z given X = 5: -0.022
Covariance of Y and Z given X = 6: -0.023
Covariance of Y and Z given X = 7: -0.051
Covariance of Y and Z given X = 8: -0.134
Covariance of Y and Z given X = 9: -0.097



**Output.** The dice covariance is close to 0, as independence predicts. The overall covariance between $Y$ and $Z$ is large (around 25), because both are driven by $X$. But within every single value of $X$, the covariance collapses to roughly 0. That is exactly what $Y \perp Z \mid X$ looks like: the dependence between $Y$ and $Z$ came entirely through $X$, so once $X$ is held fixed, nothing is left.

### The lecture example: height and reading ability

Height and the ability to read are not independent: very short people (babies) can't read, while adult-sized people usually can.

But condition on age. Among 20-year-olds, there should be no systematic relationship between height and reading:

$$
P(\text{can read} \mid \text{Age}=20, \text{Height}) = P(\text{can read} \mid \text{Age}=20)
$$

And among newborns, nobody reads regardless of height:

$$
P(\text{can read} \mid \text{Age}=0, \text{Height}) = P(\text{can read} \mid \text{Age}=0) = 0
$$

Reading ability and height are not independent, but they are (close to) independent conditional on age. Equivalently, the literacy rate among 6-foot 20-year-olds is about the same as among 5-foot 20-year-olds.

The next cell simulates this world so we can see it in data — and so we can see what "controlling for age" in a regression does.

In [ ]:
np.random.seed(15)
n = 5000                                            # number of simulated people

# age= common cause: it drives both height and the chance of being able to read.
age = np.random.randint(0, 31, n)                   # ages 0 to 30

# height in cm : about 50 cm at birth, growing 6.5 cm a year until 18, then flat; plus person-specific noise.
height = 50 + 6.5 * np.minimum(age, 18) + np.random.normal(0, 8, n)

# p of reading depends ONLY on age (an S-shaped curve centred at 5.5 years old).
p_read = 1 / (1 + np.exp(-2 * (age - 5.5)))
can_read = np.random.binomial(1, p_read)            # 1 = can read, 0 = cannot; one coin flip per person

people = pd.DataFrame({'age': age, 'height': height, 'can_read': can_read})

# 1) ignoring age: readers vs. non-readers look completely different in height.
print("Average height (cm) by reading ability, everyone pooled:")
print(people.groupby('can_read')['height'].mean().round(1), "\n")

# 2) holding age fixed: at ages where some children read and some don't, compare their heights.
for a in [5, 6]:
    same_age = people[people['age'] == a]
    print(f"Age {a}: average height by reading ability:",
          same_age.groupby('can_read')['height'].mean().round(1).to_dict())

Average height (cm) by reading ability, everyone pooled:
can_read
0     66.7
1    146.9
Name: height, dtype: float64 

Age 5: average height by reading ability: {0: 82.0, 1: 83.0}
Age 6: average height by reading ability: {0: 90.4, 1: 89.3}


**Code Explained**

- `age = np.random.randint(0, 31, n)` gives each of the 5,000 people an age between 0 and 30.
- `height = 50 + 6.5 * np.minimum(age, 18) + np.random.normal(0, 8, n)`: `np.minimum(age, 18)` caps age at 18, so height grows by 6.5 cm a year until adulthood and then stops. The normal noise (standard deviation 8 cm) makes people of the same age differ in height for reasons that have nothing to do with reading.
- `p_read = 1 / (1 + np.exp(-2 * (age - 5.5)))` is a logistic (S-shaped) curve: almost 0 for toddlers, about 0.5 at 5½, almost 1 by age 8. Crucially, **height does not appear in this formula** — given age, reading is unrelated to height.
- `np.random.binomial(1, p_read)` flips one weighted coin per person, landing on 1 with that person's probability `p_read`.
- `pd.DataFrame({...})` stores the three columns in a table called `people`.
- `people.groupby('can_read')['height'].mean()` splits the table into readers and non-readers and averages height within each group. `.round(1)` keeps one decimal place.
- The `for` loop filters to a single age (`people['age'] == a`) and repeats the same comparison within that age. We pick ages 5 and 6 because those are the ages where there is a real mix of readers and non-readers to compare.

**Output** Pooled, readers are dramatically taller than non-readers (roughly 147 cm vs. 67 cm), because readers are mostly older. Within age 5 or age 6, readers and non-readers are about the same height (within about 1 cm). The pooled gap was entirely an age gap.

In [4]:

# (a) regress reading on height alone.
m_pooled = smf.ols('can_read ~ height', data=people).fit()

# (b) regress reading on height AND age. C(age) turns age into a set of 0/1 indicators,
#     one for each age, so we compare people only against others of exactly the same age.
m_age = smf.ols('can_read ~ height + C(age)', data=people).fit()

print(f"Height coefficient, no control:      {m_pooled.params['height']:.5f}  (SE {m_pooled.bse['height']:.5f})")
print(f"Height coefficient, controlling age: {m_age.params['height']:.5f}  (SE {m_age.bse['height']:.5f})")

Height coefficient, no control:      0.00763  (SE 0.00009)
Height coefficient, controlling age: -0.00014  (SE 0.00022)


**Code Explained**

- `smf.ols('can_read ~ height', data=people)` sets up an OLS regression with `can_read` as the outcome and `height` as the only regressor, using the columns of `people`. `.fit()` actually estimates it and returns a results object.
- In `'can_read ~ height + C(age)'`, the `C(...)` wrapper tells statsmodels to treat `age` as **categorical**: it creates one dummy variable for each age (leaving one out as the baseline). This lets each age have its own average reading rate, without forcing that relationship to be a straight line.
- `.params['height']` pulls out the estimated coefficient on height, and `.bse['height']` pulls out its standard error.

**Output** Without the control, the height coefficient is about 0.0076 and hugely significant: each extra centimetre "predicts" a higher chance of reading. With age dummies, the coefficient falls to essentially zero and is statistically insignificant. Controlling for age absorbs the link between height and reading, because that link ran entirely through age. 

## 3. Control variables

Say we have something that feels like a natural experiment, but isn't quite. Maybe the treatment is random with respect to $U$ - conditional on some third variable. If so, we call that third variable a **control variable**, and we can recover a causal estimate.

The crucial implication for using controls:

$$
E[U \mid S, C] = E[U \mid C] \quad\Longrightarrow\quad E[U \mid S, C=c] = E[U \mid C=c]
$$

For the subpopulation defined by any specific value of the control, $S$ is as good as random within that group.

**Lecture example** Suppose the lottery administrators gave rural postcodes double the chance of winning, as a promotion for farmers. The natural experiment is ruined: winning is no longer random with respect to $U$, because rural households differ from urban ones in many ways that affect spending. But if we include being rural as a control, we're fine as winning is random conditional on being rural or not.

## 4. Multiple linear regression

There are several ways to use a conditional-independence argument for causal inference. We start with **multiple linear regression**, which is how control variables were first implemented in the literature.

Multiple linear regression is simply a regression with more than one regressor. As with simple regression, it has both a **descriptive** and a **causal** interpretation, and we notate them differently to keep them apart.

### 4.1 Descriptive multiple linear regression

$$
Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_k X_k + \varepsilon
$$

This has the same interpretation as descriptive simple regression: it is the **best linear predictor** of $Y$ given several $X$ variables. The coefficients solve

$$
\min_{b_0, b_1, \dots, b_k} \; E\big[(Y - (b_0 + b_1 X_1 + \dots + b_k X_k))^2\big]
$$

— still the best linear predictor, just in a higher-dimensional space. The residual is defined as the prediction error,

$$
\varepsilon \equiv Y - (\beta_0 + \beta_1 X_1 + \dots + \beta_k X_k).
$$

To identify the parameters we need

$$
E[\varepsilon] = E[\varepsilon X_i] = 0 \quad \forall\, i \in \{1, \dots, k\}.
$$

Under the descriptive interpretation these conditions hold automatically — they are just the first-order conditions of the minimization problem above. So we can (basically) always consistently estimate descriptive regression parameters. The next cell checks this numerically.

In [5]:
# Pull the residuals (prediction errors) out of the height regression with age dummies.
resid = m_age.resid

print(f"Mean of the residuals:               {resid.mean():.2e}")
print(f"Mean of residual × height:           {(resid * people['height']).mean():.2e}")
print(f"Mean of residual × (age == 10 dummy): {(resid * (people['age'] == 10)).mean():.2e}")

Mean of the residuals:               2.51e-15
Mean of residual × height:           3.27e-14
Mean of residual × (age == 10 dummy): 3.17e-15


**Code Explained**

- `m_age.resid` is the vector of residuals $\hat\varepsilon_i = Y_i - \hat Y_i$ from the regression we fit earlier — one number per person.
- `resid.mean()` is the sample version of $E[\varepsilon]$.
- `(resid * people['height']).mean()` multiplies each residual by that person's height and averages: the sample version of $E[\varepsilon X_i]$ for the height regressor.
- `(people['age'] == 10)` is a True/False column that Python treats as 1/0 when multiplied, so the third line checks $E[\varepsilon X_i]$ for one of the age dummies.
- `:.2e` prints the number in scientific notation with two decimals.

**Output:** All three numbers are on the order of $10^{-12}$ or smaller — zero up to computer rounding. OLS forces these conditions to hold in the sample. That is why the descriptive interpretation is always available; the hard part, below, is justifying a causal one.

### 4.2 Causal multiple linear regression

The most common causal use of multiple regression is to include control variables. Suppose the all-causes model is linear for treatment $S$ and outcome $Y$:

$$
Y(S, U) = \alpha_0 + \alpha_1 S + U
$$

Suppose $S \not\perp U$ — e.g. the effect of winning the lottery $S$ on purchases $Y$, in a lottery that favours rural postcodes. But suppose there is a control $C$ with

$$
S \perp U \mid C \quad\Longrightarrow\quad E[U \mid S, C] = E[U \mid C].
$$

Assume further that $E[U \mid C]$ is linear (automatic if $C$ is binary). Define a new error term $V$:

$$
\begin{aligned}
V &\equiv U - E[U \mid S, C] \\
  &= U - E[U \mid C] && (U \perp S \mid C) \\
  &= U - (\gamma_0 + \gamma_1 C) && (E[U \mid C] \text{ linear}) \\
\Rightarrow\; U &= V + \gamma_0 + \gamma_1 C
\end{aligned}
$$

Substitute into the causal model:

$$
\begin{aligned}
Y(S, U) &= \alpha_0 + \alpha_1 S + U \\
        &= \alpha_0 + \alpha_1 S + V + \gamma_0 + \gamma_1 C \\
        &= \underbrace{(\alpha_0 + \gamma_0)}_{\equiv\,\tilde\alpha_0} + \alpha_1 S + \underbrace{\gamma_1}_{\equiv\,\tilde\alpha_2} C + V \\
        &= \tilde\alpha_0 + \alpha_1 S + \tilde\alpha_2 C + V
\end{aligned}
$$

The causal effect $\alpha_1$ is still in the model. Intuitively, including $C$ absorbs the connection between $S$ and $U$: the "residual" variation in $U$, namely $V$, is independent of $S$. Put differently, the control accounts for differences in outcomes between people with different values of $C$; once those differences are accounted for, the remaining variation in $S$ is random with respect to the remaining variation in $U$.

### 4.3 Identification

For multiple regression, the parameters are identified if the error is mean-zero and uncorrelated with every regressor. Here that means

$$
E[V] = E[SV] = E[CV] = 0.
$$

Each holds in our construction:

$$
E[V] = E\big[U - E[U \mid S, C]\big] = E[U] - E\big[E[U \mid S, C]\big] = E[U] - E[U] = 0 \quad \text{(LIE)}
$$

$$
\begin{aligned}
E[VS] &= E\big[(U - E[U \mid S, C])\,S\big] \\
      &= E[US] - E\big[E[U \mid S, C]\,S\big] \\
      &= E[US] - E\big[E[US \mid S, C]\big] && \text{(}S\text{ is known given }S, C\text{)} \\
      &= E[US] - E[US] = 0 && \text{(LIE)}
\end{aligned}
$$

The exact same argument gives $E[VC] = 0$. So we can consistently estimate $\alpha_1$ — the linear, causal effect of $S$ on $Y$.

**Caution:** we also get an estimate of $\tilde\alpha_2$. This is not a causal parameter — we only ever defined it descriptively ($\tilde\alpha_2 = \gamma_1$, the slope of $E[U \mid C]$). It describes the relationship between $Y$ and $C$, not the effect of $C$.

**Multiple controls.** If $S \perp U \mid C_1, \dots, C_k$, the same argument works with all the controls together. E.g. if the lottery favours rural postcodes *and* more populous postcodes, control for both simultaneously.

### 4.4 What we need, in summary

1. Control variable(s) $C$ such that $S \perp U \mid C$.
2. A linear relationship between $Y$ and $S$ (automatic if $S$ is binary).
3. $E[U \mid C]$ linear (automatic if $C$ is binary, or if we control with a full set of indicator variables).

## 5. Simulation: the postcode lottery with a rural bonus

Build the lecture's example so we know the true answer and can check what each estimator recovers.

- $S$ = won the lottery (1/0). Urban postcodes win with probability 10%; rural postcodes with 20% (the "double chance").
- $Y$ = household purchases over the following year, in euros.
- True causal effect: $\alpha_1 = 3{,}000$ — winning raises purchases by €3,000 for everyone.
- $U$ = everything else that affects purchases. Rural households spend less (−€3,000), and household income also matters; rural households earn less on average. Plus idiosyncratic noise.
- $C$ = rural (1/0).

Because rural households are both more likely to win and spend less for other reasons, $S \not\perp U$. But within rural and within urban postcodes, winning is a pure lottery, so $S \perp U \mid C$.

In [7]:
np.random.seed(2026)
n = 20000                                                    # number of households

#  control variable: 30% of households live in rural postcodes.
rural = np.random.binomial(1, 0.3, n)

# hhld income: urban households average €50,000, rural ones €40,000, with spread around that.
income = np.random.normal(50000 - 10000 * rural, 8000)

# treatment: winning. Urban chance 10%, rural chance 20%. Nothing else affects who wins.
win = np.random.binomial(1, 0.10 + 0.10 * rural)

# U, "everything else": a rural spending gap, an income effect, and individual noise.
U = -3000 * rural + 0.1 * (income - 50000) + np.random.normal(0, 2000, n)

#  causal model Y(S, U) = alpha_0 + alpha_1 * S + U, with alpha_1 = 3,000.
purchases = 20000 + 3000 * win + U

lottery = pd.DataFrame({'purchases': purchases, 'win': win, 'rural': rural, 'U': U})
lottery.head()

,purchases,win,rural,U
0,20876.295565,0,0,876.295565
1,21481.691347,0,0,1481.691347
2,15034.155057,0,1,-4965.844943
3,17103.126785,0,0,-2896.873215
4,25834.214447,1,0,2834.214447


**Code Explained**

- `rural = np.random.binomial(1, 0.3, n)` gives each household a 30% chance of being rural (1) and 70% of being urban (0).
- `income = np.random.normal(50000 - 10000 * rural, 8000)`: the mean argument is an array, so each household gets its own mean — €50,000 if urban (`rural` = 0) and €40,000 if rural — with a standard deviation of €8,000.
- `win = np.random.binomial(1, 0.10 + 0.10 * rural)`: the win probability is 0.10 for urban and 0.20 for rural. Only `rural` enters this line, which is exactly what makes winning random *conditional on* rural.
- `U = -3000 * rural + 0.1 * (income - 50000) + np.random.normal(0, 2000, n)` builds "everything else": rural households spend €3,000 less; each extra euro of income above €50,000 adds 10 cents of spending; and there's €2,000 of household-specific noise. In real data we would never observe `U` — we keep it here only so we can check the assumption directly.
- `purchases = 20000 + 3000 * win + U` is the all-causes model with $\alpha_0 = 20{,}000$ and $\alpha_1 = 3{,}000$.
- `lottery.head()` shows the first five rows of the table as a quick sanity check.

### Checking the assumption directly: $E[U \mid S, C] = E[U \mid C]$

Since this is a simulation, we can look at $U$. If conditional independence holds, then the average of $U$ should differ between rural and urban households (that's the confounding), but within each, winners and losers should have the same average $U$.

In [8]:
# Average of U for each combination of (rural, win).
lottery.groupby(['rural', 'win'])['U'].mean().round(0)

rural  win
0      0        13.0
       1        50.0
1      0     -3988.0
       1     -4060.0
Name: U, dtype: float64

**Code** `groupby(['rural', 'win'])` splits the data into four cells — urban losers, urban winners, rural losers, rural winners — and `['U'].mean()` averages $U$ within each. `.round(0)` rounds to whole euros.

**Output** Urban households have $E[U]$ around €0 and rural households around −€4,000, so $U$ clearly depends on $C$. But within urban, winners and losers have about the same $E[U]$, and the same within rural. That is $E[U \mid S, C] = E[U \mid C]$ in action. (Notice the rural gap is about −€4,000, not −€3,000: it also includes the lower average income of rural households. Keep that in mind for the coefficient on `rural` below.)

### Naive comparison vs. regression with the control

In [9]:
# (1) Naive difference in means: average purchases of winners minus losers.
naive_diff = (lottery.loc[lottery['win'] == 1, 'purchases'].mean()
              - lottery.loc[lottery['win'] == 0, 'purchases'].mean())
print(f"Naive difference in means: {naive_diff:,.0f}\n")

# (2) Simple regression of purchases on winning (identical to the naive difference).
m_naive = smf.ols('purchases ~ win', data=lottery).fit()
print("Regression WITHOUT the control:")
print(m_naive.summary().tables[1], "\n")

# (3) Multiple regression controlling for rural.
m_ctrl = smf.ols('purchases ~ win + rural', data=lottery).fit()
print("Regression WITH the control:")
print(m_ctrl.summary().tables[1])

Naive difference in means: 2,222

Regression WITHOUT the control:
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    1.89e+04     21.377    884.276      0.000    1.89e+04    1.89e+04
win         2221.5783     59.063     37.614      0.000    2105.810    2337.346

Regression WITH the control:
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   2.002e+04     18.721   1069.297      0.000       2e+04    2.01e+04
win         2988.8251     45.412     65.816      0.000    2899.815    3077.836
rural      -4018.4127     33.359   -120.458      0.000   -4083.800   -3953.026


**Code Explained**

- `lottery.loc[lottery['win'] == 1, 'purchases']` selects the `purchases` column for rows where `win` equals 1; `.mean()` averages it. We subtract the same quantity for losers. The `:,.0f` format prints the result with thousands separators and no decimals.
- `smf.ols('purchases ~ win', ...)` regresses purchases on the winning indicator alone. With a single binary regressor, the slope is mathematically identical to the difference in means above.
- `.summary()` produces the full statsmodels report; `.tables[1]` keeps only the coefficient table (estimates, standard errors, t-statistics, p-values and 95% confidence intervals), which is the part we need.
- `smf.ols('purchases ~ win + rural', ...)` adds `rural` as a control: this is the estimating equation $Y = \tilde\alpha_0 + \alpha_1 S + \tilde\alpha_2 C + V$ from Section 4.

**Output:**

- **Without the control**, the estimated effect is about €2,200 — well below the true €3,000. Winners are disproportionately rural, and rural households spend less for other reasons, so the naive comparison mixes the lottery effect with the rural spending gap.
- **With the control**, the coefficient on `win` is about €2,990, with a 95% confidence interval covering €3,000. Controlling for rural recovers $\alpha_1$.
- The coefficient on `rural` is about −€4,000. This is $\tilde\alpha_2$, and it is not the causal effect of living in a rural area (which we set to −€3,000): it also soaks up the income difference between rural and urban households. Exactly as the lecture says, it is a descriptive parameter.

## 6. When to use controls

In the lottery example, the argument for $S \perp U \mid C$ came from knowing where $S$ comes from. Our first real example uses the same kind of contextual argument.

**The question.** Why do wealthy parents have wealthy children? Two broad explanations — classic nature vs. nurture:

- wealthy parents give their children better resources, and those resources help them get ahead;
- wealthy parents have genes that lead to wealth accumulation, and pass those genes on.

Let $S$ = parental net worth and $Y(S, U)$ = the child's net worth, where $U$ is "everything else" — and we explicitly count parents' genes as part of $U$, because we want the effect of parental wealth isolated from genetic links. In general $S \not\perp U$: wealthy parents may disproportionately have genetic characteristics they pass on to their kids.

**An (almost) natural experiment.** After the Korean War, many Korean orphans were adopted internationally, with Norway a major centre (especially 1965–1968). A centralized agency handled adoptions: approved families were matched with children in the order applications were approved and the order children became available. Adopters could not specify any characteristics of the child (not even gender). This breaks any link between the child's genes and the parents' wealth.

**The one source of non-randomness.** Matching depended on application timing. Suppose older prospective parents were more eager and applied earlier — and older applicants tend to be further into their careers and wealthier. Suppose older children were given priority. Age at adoption is plausibly part of $U$ (more of life spent with a family rather than in group care could change adult outcomes). Then wealthier families get older kids, and $S \not\perp U$.

**The fix.** The only thing that meaningfully affects matching is the timing of the application. Within a given batch of applications, which parents get which child should be random. So with $C$ = year of application, it is reasonable to claim

$$
S \perp U \mid C.
$$

This argument appeals directly to knowledge of how $S$ is "allocated" in this context: we take advantage of a quasi-random process, while adjusting for the one factor that could plausibly disturb it.

### Simulating the adoption setting

We build a simplified version: four application years, 300 families per year. Earlier years have wealthier parents *and* older children; within each year, children are assigned to families at random. The true causal effect is that each extra \$1,000 of parental wealth raises the child's adult wealth by \$300 (so $\alpha_1 = 0.30$ with both measured in thousands). Older age at adoption lowers adult wealth.

In [10]:
np.random.seed(1965)
years = [1965, 1966, 1967, 1968]
batches = []                                     # one table per application year, then stack them

for i, year in enumerate(years):                 # i = 0, 1, 2, 3 counts the years; year is the actual year
    m = 300                                      # families (and children) in this year's batch

    # parents who applied earlier are wealthier on average (thousands of USD).
    parent_wealth = np.random.normal(250 - 40 * i, 80, m)

    # children matched earlier are older on average (years); clip stops ages going below 0.
    child_age = np.clip(np.random.normal(3.0 - 0.8 * i, 0.6, m), 0, None)

    # gender is unrelated to anything (here, 60% girls).
    girl = np.random.binomial(1, 0.6, m)

    # within the batch, children are lined up with families at random: shuffle the order of the children.
    np.random.shuffle(child_age)

    batches.append(pd.DataFrame({'year': year,
                                 'parent_wealth': parent_wealth,
                                 'child_age': child_age,
                                 'girl': girl}))

adoptees = pd.concat(batches, ignore_index=True)      # stack the four batches into one table

# child's adult wealth (thousands): true effect of parental wealth is 0.30; older adoptees do worse.
adoptees['child_wealth'] = (40 + 0.30 * adoptees['parent_wealth']
                            - 25 * adoptees['child_age']
                            + np.random.normal(0, 40, len(adoptees)))

adoptees.groupby('year')[['parent_wealth', 'child_age']].mean().round(2)

,parent_wealth,child_age
year,,
1965,251.55,2.98
1966,209.47,2.27
1967,174.39,1.36
1968,123.81,0.73


### Testing the control-variable assumption

When we make identifying assumptions, we want to test them to the extent possible. We're claiming that once we control for $C$, the remaining variation in other factors $U$ is random with respect to $S$. So if we regress components of $U$ (other attributes) on $S$, controlling for $C$, there should be no systematic relationship.

$$
\text{Child attribute} = \beta_0 + \beta_1\,\text{parent net worth} + \beta_2\,\text{year of application}
$$

and look for estimates of $\beta_1$ that are very close to 0 and statistically insignificant. In their Table 2, the coefficients of child's age at adoption and gender on parental net wealth are tiny ($-0.002$ and $0.005$) with standard errors larger than the estimates, and joint F-tests have p-values of 0.54 and 0.96.

Below we run the same balance test on our simulated data — first without the year control, then with it.

In [11]:
for attribute in ['child_age', 'girl']:
    no_ctrl   = smf.ols(f'{attribute} ~ parent_wealth', data=adoptees).fit()
    with_ctrl = smf.ols(f'{attribute} ~ parent_wealth + C(year)', data=adoptees).fit()

    print(f"Dependent variable: {attribute}")
    print(f"  without year control: beta_1 = {no_ctrl.params['parent_wealth']:.5f}, "
          f"p-value = {no_ctrl.pvalues['parent_wealth']:.3f}")
    print(f"  with year control:    beta_1 = {with_ctrl.params['parent_wealth']:.5f}, "
          f"p-value = {with_ctrl.pvalues['parent_wealth']:.3f}\n")

Dependent variable: child_age
  without year control: beta_1 = 0.00467, p-value = 0.000
  with year control:    beta_1 = 0.00011, p-value = 0.614

Dependent variable: girl
  without year control: beta_1 = -0.00003, p-value = 0.859
  with year control:    beta_1 = 0.00002, p-value = 0.921



**Output**

- **Child age, no control:** $\hat\beta_1 \approx 0.0047$ with a p-value of essentially 0. Wealthier parents got older children — the confounding we built in (it runs through application year).
- **Child age, with year control:** $\hat\beta_1 \approx 0.0001$ and insignificant. Within a year, parental wealth tells us nothing about the child's age — the identifying assumption passes the test, as in the paper.

### The limits of testing

These tests share the limits of balance tests in experiments: **we cannot observe unobservables.** If some unobserved component of $U$ is associated with $S$ even conditional on $C$, this kind of test cannot detect it.

We have to rest on our understanding of the context to address these fears. Based on what we know about how the Norwegian matching worked, we can be fairly confident child-parent matching was (conditionally) random.

### Estimating the causal effect

Having tested the assumption, we estimate

$$
\text{Child net worth} = \beta_0 + \beta_1\,\text{parent net worth} + \beta_2\,\mathbb{1}[\text{1965}] + \beta_3\,\mathbb{1}[\text{1966}] + \beta_4\,\mathbb{1}[\text{1967}]
$$

**Why indicators for each year, rather than year as a single number?** Our derivation needs $E[U \mid C]$ to be linear in whatever we put in the regression. With a full set of year indicators, the regression gives each year its own intercept, so $E[U \mid \text{year}]$ is exactly captured no matter how it varies from year to year — there is no need to assume a straight-line trend across years. (`C(year)` in statsmodels does this for us, with 1965 as the omitted baseline instead of 1968; the choice of baseline does not change $\hat\beta_1$.)

In [12]:
m_adopt_naive = smf.ols('child_wealth ~ parent_wealth', data=adoptees).fit()
m_adopt_ctrl  = smf.ols('child_wealth ~ parent_wealth + C(year)', data=adoptees).fit()

print("Without year control:")
print(m_adopt_naive.summary().tables[1], "\n")
print("With year-of-application indicators:")
print(m_adopt_ctrl.summary().tables[1])

Without year control:
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        12.5773      2.978      4.224      0.000       6.735      18.420
parent_wealth     0.2046      0.014     14.532      0.000       0.177       0.232

With year-of-application indicators:
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         -37.3883      4.471     -8.362      0.000     -46.160     -28.616
C(year)[T.1966]    20.2516      3.473      5.832      0.000      13.438      27.065
C(year)[T.1967]    38.6650      3.605     10.726      0.000      31.592      45.738
C(year)[T.1968]    56.0570      3.913     14.324      0.000      48.379      63.735
parent_wealth       0.3164      0.015     21.151      0.000       0.287       0.346


**Code** The first line regresses child wealth on parental wealth alone; the second adds the year indicators. We print just the coefficient table for each.

**Output** Without the control, the parental-wealth coefficient is about 0.20 — biased downward from the true 0.30, because wealthier parents (early applicants) systematically received older children, who are less well off as adults. With the year indicators, the estimate is about 0.32, with a confidence interval covering 0.30. The coefficients on the `C(year)` terms are descriptive — they summarize how child wealth differs across application cohorts, not the "effect" of a year.


**The takeaway on method:** the use of the control is motivated by understanding *how $S$ was assigned*. The underlying argument rests on a piece of quasi-randomness (the matching process), and the control just sharpens that argument.

## 7. Kitchen-sink regression

Sometimes people control for many characteristics and claim (or imply) that this identifies a causal parameter. This is called **kitchen-sink regression**. It does not appeal to any source of quasi-randomness. For it to work, $S$ must be random with respect to $U$ **conditional on all of those characteristics** — a very specific claim that can be hard to defend under scrutiny.

**The question.** The military is one of the largest employers in the US. Military service could plausibly affect lifetime income through direct pay, more accessible higher education, easier future employment (employer perceptions, networks), and changes to personality and habits.

Let $S = 1$ if someone serves and $0$ otherwise, and $Y(S, U)$ = income. Generally $S \not\perp U$: whether someone serves is associated with personality traits, values, career and life goals, and taste for order and discipline — all of which plausibly affect income.

**Angrist's approach.** He studies applicants to the military and compares later incomes between those who enlist and those who don't (either rejected, or accepted but declined). He controls for **age, years of schooling, application year, and AFQT score** (the military's standardized test). (He also uses instrumental variables; we'll come back to that later.) For the regression to be causal we need

$$
S \perp U \mid \text{age, yrs. of school, app. year, AFQT score}.
$$

**Thought experiment.** Imagine two 20-year-old high-school graduates who both applied in 1975 and got similar AFQT scores. One enlists; one doesn't. Are they identical apart from service? Probably not:

- the military might have rejected one for personality or character traits, or behaviour during the application process;
- one might have declined because they disliked the idea of serving, or because they had something else to do (a job or education opportunity).

Personality, taste for structure, and outside options plausibly affect both $S$ and income — they're part of $U$. If so, $S \not\perp U \mid$ the controls, and the coefficient on service cannot be interpreted causally.

Also note: **controlling for more isn't always better.** Unless you truly achieve $S \perp U \mid C$, adding controls can even *increase* bias.

In [13]:
np.random.seed(1998)
n = 20000                                                   # military applicants

# Observed characteristics (the "kitchen sink").
age      = np.random.randint(17, 23, n)                     # 17 to 22
school   = np.random.randint(10, 15, n)                     # 10 to 14 years of schooling
app_year = np.random.randint(1976, 1981, n)                 # application years 1976 to 1980
afqt     = np.clip(50 + 5 * (school - 12) + np.random.normal(0, 15, n), 1, 99)   # test percentile, 1-99

# Unobserved: quality of outside options (a good civilian job offer, a college place, etc.).
outside = np.random.normal(0, 1, n)

# Who serves: higher AFQT helps (the military accepts them); good outside options make people decline.
p_serve = 1 / (1 + np.exp(-(0.3 + 0.02 * (afqt - 50) - 1.0 * outside)))
serve = np.random.binomial(1, p_serve)

# Later income. TRUE causal effect of service = +2,000. Outside options also raise income.
income = (20000 + 2000 * serve
          + 300 * (age - 19) + 1500 * (school - 12) + 100 * (afqt - 50)
          + 4000 * outside
          + np.random.normal(0, 5000, n))

applicants = pd.DataFrame({'income': income, 'serve': serve, 'age': age, 'school': school,
                           'app_year': app_year, 'afqt': afqt, 'outside': outside})
print(f"Share who serve: {serve.mean():.2f}")

Share who serve: 0.56


**Code explained**

- `age`, `school`, `app_year`: uniformly drawn whole numbers in the given ranges (remember the upper bound in `randint` is excluded).
- `afqt = np.clip(50 + 5 * (school - 12) + noise, 1, 99)`: AFQT percentile rises 5 points per extra year of school above 12, plus noise, then `np.clip` keeps it between 1 and 99.
- `outside = np.random.normal(0, 1, n)` is the **unobserved** quality of each applicant's civilian alternatives. We create it so we can see the bias it causes, but we'll pretend the researcher cannot see it.
- `p_serve` is a logistic probability of serving. The `+ 0.02 * (afqt - 50)` term means higher scorers are more likely to be accepted; the `- 1.0 * outside` term means people with better outside options are more likely to decline. `np.random.binomial(1, p_serve)` turns those probabilities into actual 0/1 service decisions.
- `income = ...`: a baseline of \$20,000, a **true service effect of +\$2,000**, effects of age, schooling and AFQT, and +\$4,000 per standard deviation of outside options, plus noise.
- The final line prints the share of applicants who serve, as a quick sanity check (about 56%).

In [14]:
controls = 'age + school + C(app_year) + afqt'

m_mil_naive   = smf.ols('income ~ serve', data=applicants).fit()
m_mil_sink    = smf.ols(f'income ~ serve + {controls}', data=applicants).fit()
m_mil_oracle  = smf.ols(f'income ~ serve + {controls} + outside', data=applicants).fit()

print("True causal effect of service: 2,000\n")
for label, model in [('No controls', m_mil_naive),
                     ('Kitchen sink (age, school, app. year, AFQT)', m_mil_sink),
                     ('Kitchen sink + unobserved outside options', m_mil_oracle)]:
    low, high = model.conf_int().loc['serve']
    print(f"{label:<45} {model.params['serve']:>8,.0f}   95% CI [{low:,.0f}, {high:,.0f}]")

True causal effect of service: 2,000

No controls                                       -477   95% CI [-675, -280]
Kitchen sink (age, school, app. year, AFQT)     -1,217   95% CI [-1,391, -1,043]
Kitchen sink + unobserved outside options        1,988   95% CI [1,833, 2,142]



**Output**

- **No controls:** about −\$480 — the wrong sign.
- **Kitchen sink:** about −\$1,200, *further* from the truth. Adding observed controls did not fix the problem — it made it worse. Here's why: AFQT pushes people into service, so among applicants with the same AFQT score, the remaining variation in who serves is driven more heavily by outside options, the unobserved confounder. More controls aren't always better.
- **Adding the unobserved confounder:** about \$1,990, matching the truth. The regression method works fine — *if* you control for the right thing. The kitchen-sink approach fails because the right thing is unobservable.

**How should we read a kitchen-sink coefficient when we don't believe the identifying assumption?** As a **descriptive** regression parameter. It describes some combination of the causal effect of service and the influence of all the non-random factors that determine who serves. It's not obvious what to do with that information.

## 8. Summary

- **Conditional independence** $S \perp U \mid C$ means that within groups sharing the same value of $C$, treatment is as good as random. Its key implication is $E[U \mid S, C] = E[U \mid C]$.
- Under that assumption, plus linearity of $Y$ in $S$ and of $E[U \mid C]$, the regression $Y = \tilde\alpha_0 + \alpha_1 S + \tilde\alpha_2 C + V$ satisfies $E[V] = E[SV] = E[CV] = 0$, so OLS consistently estimates the causal effect $\alpha_1$. The coefficient on the control, $\tilde\alpha_2$, is **descriptive only**.
- Using indicator variables for the control (as with application year) makes $E[U \mid C]$ linear automatically.
- **Test what you can:** regress observed components of $U$ on $S$ controlling for $C$, and look for coefficients near zero. But no test can rule out unobserved confounders — that has to come from understanding the context.
- **Credible controls come from knowing how $S$ was assigned** (the lottery rule, the adoption queue). **Kitchen-sink regression** — controlling for everything available and hoping — rests on a much stronger claim, and adding controls can even increase bias.

## 9. Exercises

Run every cell above before starting the code exercises, since they reuse the `lottery` and `applicants` data frames. Click **Show answer** under each question to check your work.

**Q1** In Section 5, the coefficient on `rural` is about −€4,000, but we built the simulation so that living in a rural area lowers spending by €3,000. Where does the extra −€1,000 come from, and why doesn't it cause a problem for the coefficient on `win`?

<details>
<summary><b>Show answer</b></summary>

Rural households also earn about €10,000 less on average, and each euro of income adds 10 cents of spending. So the rural coefficient picks up the €3,000 direct gap plus about €1,000 through income. It is $\tilde\alpha_2 = \gamma_1$, the slope of $E[U \mid C]$ — a descriptive parameter, not the causal effect of being rural.

This doesn't affect the coefficient on `win` because the derivation only needs $E[U \mid S, C] = E[U \mid C]$. Income is part of $U$, but winning is random within rural and within urban households, so income is unrelated to winning once we control for rural. Whatever $C$ soaks up from $U$ doesn't matter, as long as what's left over ($V$) is unrelated to $S$.
</details>

**Q2** In the adoption example, suppose the agency steered "cuter" children toward wealthier families, and cuteness raises adult wealth. Why can't the balance test in Section 6 detect this, and in which direction would it bias the estimated effect of parental wealth?

<details>
<summary><b>Show answer</b></summary>

The balance test can only check *observed* components of $U$, like age and gender. Cuteness isn't recorded, so we can't regress it on parental wealth. This is the same limitation as balance tests in experiments: we cannot observe unobservables.

The bias would be **upward**. Within an application year, wealthier parents would get children who are more likely to become wealthy for reasons unrelated to their parents' money. The regression would attribute that extra wealth to parental wealth, overstating its effect. Our only defence is knowledge of the context — here, the documented first-come, first-served matching process.
</details>

**Q3** Suppose richer households win the lottery more often (say they buy more tickets), on top of the rural bonus. The cell below re-simulates Section 5 with only that change; the true effect of winning is still €3,000.

1. Regress `purchases` on `win` and `rural`. How far is the estimate from 3,000, and in which direction?
2. Add `income` as a control. What happens, and why?

In [ ]:
np.random.seed(7)
n_q3 = 20000

rural_q3  = np.random.binomial(1, 0.3, n_q3)
income_q3 = np.random.normal(50000 - 10000 * rural_q3, 8000)

# Winning: 10% urban / 20% rural as before, plus 1 percentage point per €1,000 of income above €50,000.
p_win_q3 = np.clip(0.10 + 0.10 * rural_q3 + 0.00001 * (income_q3 - 50000), 0, 1)
win_q3   = np.random.binomial(1, p_win_q3)

U_q3 = -3000 * rural_q3 + 0.1 * (income_q3 - 50000) + np.random.normal(0, 2000, n_q3)
purchases_q3 = 20000 + 3000 * win_q3 + U_q3

lottery_q3 = pd.DataFrame({'purchases': purchases_q3, 'win': win_q3,
                           'rural': rural_q3, 'income': income_q3})

# 1. Control for rural only
# TODO

# 2. Control for rural and income
# TODO

<details>
<summary><b>Show answer</b></summary>

```python
m_q3_rural  = smf.ols('purchases ~ win + rural', data=lottery_q3).fit()
m_q3_income = smf.ols('purchases ~ win + rural + income', data=lottery_q3).fit()

print(f"Control for rural:          {m_q3_rural.params['win']:,.0f}")
print(f"Control for rural + income: {m_q3_income.params['win']:,.0f}")
```

Controlling for rural alone gives about **€3,560**, overstating the effect by roughly €560. Within each rural/urban group, winners are now richer than losers, and richer households spend more anyway. So $S \not\perp U \mid \text{rural}$, and the winners' extra spending from income gets attributed to winning.

Adding income gives about **€2,930**, close to the truth. Winning now depends only on rural and income, so it is random conditional on both, and $E[U \mid S, C]$ is linear in those controls. In real data, we could only trust this if we knew how winners were chosen — that is, if income really were the only extra factor.
</details>

**Q4** Using the `applicants` data frame from Section 7, estimate the coefficient on `serve` controlling for (a) age, school and application year only, and (b) AFQT only. Compare with the estimates in Section 7. Which control drives the estimate further from the truth, and why?

In [ ]:
# (a) age, school and application year only
# TODO

# (b) AFQT only
# TODO

<details>
<summary><b>Show answer</b></summary>

```python
m_q4_demo = smf.ols('income ~ serve + age + school + C(app_year)', data=applicants).fit()
m_q4_afqt = smf.ols('income ~ serve + afqt', data=applicants).fit()

print(f"Age, school, app. year: {m_q4_demo.params['serve']:,.0f}")
print(f"AFQT only:              {m_q4_afqt.params['serve']:,.0f}")
```

Age, school and application year give about **−€790**; AFQT alone gives about **−€1,240**, almost the full kitchen-sink estimate (−€1,217). So **AFQT** is the control pushing the estimate away from the truth (+€2,000).

From `p_serve`, only two things decide who serves: AFQT and the unobserved outside option. Holding AFQT fixed removes one of the two sources of variation in `serve`, so what's left is driven more heavily by outside options — the confounder that also raises income. Comparing servers and non-servers with the same AFQT therefore compares people with worse outside options to people with better ones, making service look worse. Adding a control that predicts treatment but doesn't fix the confounding can increase bias.
</details>